# ConceptGraphs Pipeline on Real Robot Data

This notebook runs the full **ConceptGraphs** open-vocabulary 3D scene-understanding pipeline on real robot trajectory recordings from our kitchen manipulation dataset.

## The full pipeline at a glance

```
RGB frame
    │
    ▼
[Stage 1] RAM → GroundingDINO → SAM
    Open-vocabulary detection + pixel-precise masks
    │
    ▼
[Stage 2] Visualization
    Annotated frames to verify detection quality
    │
    ▼
[Stage 3] OpenCLIP ViT-H-14
    512→1024-dim visual embedding per detected object crop
    │
    ▼
[Stage 4] RealSense depth → Open3D point cloud
    Each mask backprojected to 3D using real metric depth
    │
    ▼
[Stage 5] MapObjectList (the SLAM core)
    Multi-frame fusion: same object seen twice → one canonical entry
    │
    ▼
[Stage 6] LLaVA via Ollama
    Free-form natural-language caption per canonical object
    │
    ▼
[Stage 7] Portkey GPT scene graph
    Node refinement + spatial edge labelling ("pan ON stove")
    │
    ▼
[Stage 8] Save + visualize
    Static inline 3D scatter + concept-graphs interactive viewer
```

| Stage | Key model | What you get |
|---|---|---|
| 1 | RAM + GroundingDINO + SAM | Per-frame detections with pixel masks |
| 2 | supervision annotators | Annotated images for quality check |
| 3 | OpenCLIP ViT-H-14 | 1024-d visual embedding per detection |
| 4 | RealSense D435i depth | Per-object 3D point cloud + bbox |
| 5 | MapObjectList | De-duplicated canonical object list |
| 6 | LLaVA (Ollama) | Natural-language caption per object |
| 7 | GPT via Portkey | Scene graph: nodes + spatial edges |
| 8 | Open3D / pkl.gz | Persisted map + visualization |


## 0. Setup - imports, paths, device, and LLM client

This first block does two things that **must happen before any other import**:

### Why sys.path manipulation comes first

Python resolves module names by searching `sys.path` in order. Several packages used here (`reflect`, `conceptgraph`, `ram`, `groundingdino`, `segment_anything`) are **not pip-installed globally** - they live as source trees under `reflect_clean/third_party/`. If we `import ram` before adding the right path, Python either finds nothing or finds the wrong package.

The tricky case is `segment_anything`: the Grounded-SAM repo has a top-level folder called `segment_anything/` (the repo root), and inside that the actual Python package `segment_anything/segment_anything/`. Adding the repo root to `sys.path` makes Python treat the outer folder as a namespace package and miss the inner one. The fix is to insert `segment_anything/` (the repo root, parent of the real package) *before* the GSA root.

### What each path addition enables

| Path added | Packages it exposes |
|---|---|
| `Grounded-Segment-Anything/segment_anything/` | `segment_anything` (SAM), `sam_model_registry`, `SamPredictor` |
| `Grounded-Segment-Anything/` | `ram`, `groundingdino` |
| `concept-graphs/` | `conceptgraph` |
| `reflect_clean/src/` | `reflect` |


In [ ]:
import sys, os

# External checkouts: concept-graphs and Grounded-Segment-Anything are NOT vendored
# in this repo. Clone them separately and point these env vars at the checkouts
# (defaults assume they live under third_party/). See analysis/README.md.
from reflect.core.paths import REPO_ROOT, analysis_experiment_dir
_CG_DIR      = os.environ.get("CONCEPTGRAPHS_DIR", str(REPO_ROOT / "third_party" / "concept-graphs"))
_GSA_DIR     = os.environ.get("GSA_DIR", str(REPO_ROOT / "third_party" / "Grounded-Segment-Anything"))
# SAM repo root must come before GSA root so 'segment_anything' resolves to the
# inner package (GSA root contains an outer segment_anything/ that shadows it)
_SAM_ROOT    = _GSA_DIR + "/segment_anything"
for _p in [_SAM_ROOT, _GSA_DIR, _CG_DIR]:
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.environ["GSA_PATH"]  = _GSA_DIR
os.environ["CG_FOLDER"] = _CG_DIR

# Standard library / third-party
import time, json, gc
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
import torch.nn.functional as F
from imagecodecs import imread as _zread

# REFLECT
from reflect.llm.prompter import PortkeyLLMPrompter
from reflect.perception.open_world.captioner import caption as _caption, _crop_with_mask

# ConceptGraphs / GSA
import torchvision.transforms as TS
import supervision as sv
from ram.models import ram as _ram_cls
from ram import inference_ram
from groundingdino.util.inference import Model as GroundingDINOModel
from segment_anything import sam_model_registry, SamPredictor
from conceptgraph.utils.vis import vis_result_fast, vis_result_slow_caption
import open_clip
from conceptgraph.utils.model_utils import compute_clip_features_batched
import open3d as o3d
from conceptgraph.slam.utils import create_object_pcd
from types import SimpleNamespace
from conceptgraph.slam.slam_classes import MapObjectList, DetectionList
from conceptgraph.slam.mapping import (
    compute_spatial_similarities, compute_visual_similarities,
    aggregate_similarities, merge_detections_to_objects,
)
from conceptgraph.scenegraph.GPTPrompt import GPTPrompt
import gzip, pickle
from conceptgraph.slam.cfslam_pipeline_batch import prepare_objects_save_vis
from conceptgraph.utils.vis import show_mask, show_box

In [ ]:
# Paths / constants
CG_DIR         = Path(_CG_DIR)
GSA_DIR        = Path(_GSA_DIR)
WEIGHTS_DIR    = GSA_DIR / "weights"
from reflect.core.paths import real_world_data_root

REAL_DATA_ROOT = real_world_data_root()  # honors REFLECT_DATA_ROOT

GROUNDING_DINO_CONFIG     = str(GSA_DIR / "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py")
GROUNDING_DINO_CHECKPOINT = str(WEIGHTS_DIR / "groundingdino_swint_ogc.pth")
SAM_CHECKPOINT            = str(WEIGHTS_DIR / "sam_vit_h_4b8939.pth")
RAM_CHECKPOINT            = str(WEIGHTS_DIR / "ram_swin_large_14m.pth")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PORTKEY_API_KEY = os.environ["PORTKEY_API_KEY"]  # set in your shell or .env (see .env.example)
llm = PortkeyLLMPrompter(portkey_api_key=PORTKEY_API_KEY, model="gpt-5.4", reasoning_effort="none")

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def color_for(name: str):
    rng = np.random.RandomState(abs(hash(name)) % 2**32)
    return tuple(rng.uniform(0.3, 1.0, 3).tolist())

print(f"Device     : {DEVICE}")
print(f"Weights    : {WEIGHTS_DIR}")
print(f"Real data  : {REAL_DATA_ROOT}")

## 0. Load real robot frames

### About the dataset

Each episode is a real kitchen manipulation recording captured with a **Franka robot arm** equipped with an **Intel RealSense D435i** RGB-D camera. The data is stored in the [Zarr](https://zarr.readthedocs.io/) format - an N-dimensional array store designed for large chunked datasets.

The directory structure for each episode looks like:

```
datasets/real_data/boilWater1/
    videos/
        color/        ← RGB frames as zarr chunks, one file per step
            0.0.0.0   ← step 0  (uint8, shape 720×1280×3)
            1.0.0.0   ← step 1
            ...
        depth/        ← depth frames as zarr chunks
            0.0.0     ← step 0  (uint16, millimetres)
            1.0.0     ← step 1
            ...
        color.mp4     ← full video for quick preview
    replay_buffer.zarr/
        data/
            robot_eef_pose/   ← end-effector pose at each step
            robot_joint/      ← joint angles
            action/           ← commanded actions
            ...
```

### Zarr chunks

Zarr stores arrays as independent compressed chunks, so you can load a single frame (one chunk) without reading the whole recording. With thousands of frames per episode, this avoids loading gigabytes into RAM just to get one image.

### Depth units and camera intrinsics

The depth values are **uint16 in millimetres** from the RealSense SDK (values like 1200 = 1.2 m). We divide by 1000 to convert to metres for Open3D.

**Important:** RealSense encodes *invalid* pixels (reflective surfaces, beyond range, sensor noise) with large uint16 values rather than zero. After ÷1000 these appear as physically impossible depths like 42 m or 55 m. We zero out any value above `MAX_DEPTH_M = 4.0 m`, which is the reliable operating range of the D435i at 1280×720 in an indoor kitchen scene. Zeroed pixels are automatically excluded by `create_object_pcd` (it filters `depth > 0`).

The camera intrinsics are fixed across all recordings (same physical camera):
```
K = [[914.27,   0,    647.07],
     [  0,    913.27, 356.33],
     [  0,      0,      1   ]]
```
where `fx=fy≈914 px` gives a horizontal FoV of about 70°.

### Frame sampling

We pick `n` frames evenly spaced across the full episode trajectory. Early frames typically show the initial scene configuration; later frames show the robot interacting with objects. Sampling a spread gives `MapObjectList` multiple viewpoints to work with.


In [ ]:
# Episodes to sample - (folder_name, n_frames).
EPISODES = [
    ("boilWater1",    2),
    ("putAppleBowl1", 2),
]

# RealSense D435i intrinsics (1280×720)
REAL_K = np.array([
    [914.27246, 0.0,       647.0733 ],
    [0.0,       913.2658,  356.32526],
    [0.0,       0.0,       1.0      ],
], dtype=np.float32)

MAX_DEPTH_M = 3.0   # RealSense D435i reliable range
MIN_DEPTH_M = 0.3   # RealSense D435i minimum range


def load_episode_frames(ep_name: str, n: int):
    """Evenly sample n frames (RGB + depth) from an episode."""
    color_dir = REAL_DATA_ROOT / ep_name / "videos" / "color"
    depth_dir = REAL_DATA_ROOT / ep_name / "videos" / "depth"

    # zarr chunk filenames are "{step_idx}.0.0.0" (color) / "{step_idx}.0.0" (depth)
    # sort by numeric step index
    color_files = sorted(
        [f for f in color_dir.iterdir() if not f.name.startswith(".")],
        key=lambda f: int(f.name.split(".")[0]),
    )

    out = []
    for i in range(0, len(color_files) - 1, n):
        step_idx = int(color_files[i].name.split(".")[0])
        rgb   = _zread(str(color_dir / f"{step_idx}.0.0.0"))          # uint8 (H,W,3)
        depth = _zread(str(depth_dir / f"{step_idx}.0.0")).astype(np.float32) / 1000.0
        # depth: uint16 mm → float32 m; zero out invalid/out-of-range readings
        depth[depth > MAX_DEPTH_M] = 0.0
        out.append({
            "episode": ep_name,
            "step":    step_idx,
            "rgb":     rgb,
            "depth":   depth,          # metres
            "K":       REAL_K.copy(),
        })
        valid = depth[depth > 0]
        print(f"  [{ep_name}] step={step_idx:5d}  rgb={rgb.shape}  "
              f"depth valid=[{valid.min():.2f}, {valid.max():.2f}] m  ({valid.size} px)" if valid.size else f"  [{ep_name}] step={step_idx:5d}  NO valid depth")
    return out

frames = []
for ep_name, n in EPISODES:
    frames.extend(load_episode_frames(ep_name, n))

print(f"\nTotal frames loaded: {len(frames)}")

# ── Preview ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, len(frames), figsize=(4 * len(frames), 7))
if len(frames) == 1:
    axes = axes[:, np.newaxis]
for col, f in enumerate(frames):
    axes[0, col].imshow(f["rgb"])
    axes[0, col].set_title(f"{f['episode']}  step={f['step']}", fontsize=8)
    axes[0, col].axis("off")

    depth_vis = np.clip(f["depth"], 0, 4.0)   # clip to 4 m for contrast
    im = axes[1, col].imshow(depth_vis, cmap="turbo")
    axes[1, col].set_title("depth (0-4 m)", fontsize=8)
    axes[1, col].axis("off")
    plt.colorbar(im, ax=axes[1, col], fraction=0.046, pad=0.04)

plt.suptitle("Real robot frames - RGB (top) and RealSense depth (bottom)", fontweight="bold")
plt.tight_layout()
plt.show()

## Stage 1 - Open-vocabulary detection: RAM → GroundingDINO → SAM

This is the core innovation of ConceptGraphs: replacing a fixed-vocabulary detector (like YOLO) with a three-step pipeline that can detect *any* named object.

### Step 1 of 3: RAM (Recognize Anything Model) - image → tag list

RAM is a vision-language model trained to produce a list of natural-language tags for everything present in an image. Given a kitchen scene it might output:

```
"pot | stove | kitchen | gas burner | countertop | sink | cup"
```

This is fundamentally different from classification - RAM produces a **variable-length, open-vocabulary** list rather than a single fixed-class prediction. RAM was trained on 14 million image-tag pairs, so it generalises to long-tail objects.

RAM uses a Swin-L backbone (same family as Swin Transformer used in detection) pre-trained and fine-tuned with a contrastive objective between image regions and tag tokens.

### Step 2 of 3: GroundingDINO - tag list + image → bounding boxes

GroundingDINO takes the tag list from RAM as a **text prompt** (joined by `.`) and produces grounded bounding boxes - each box comes with the text phrase that caused it.

Internally, GroundingDINO runs a dual encoder (image encoder + text encoder) with cross-attention between image patches and text tokens, so it understands *which part of the image* corresponds to *which word*. The threshold parameters control sensitivity:
- `BOX_THRESH = 0.25` - minimum confidence to keep a detected box
- `TXT_THRESH = 0.25` - minimum text-image alignment score
- NMS at `IOU_THRESH = 0.50` removes duplicate overlapping boxes

**Example**: RAM outputs `"pot | stove"`, GroundingDINO produces:
```
box [112, 85, 340, 290]  → "pot"    conf=0.72
box [0,   200, 640, 480] → "stove"  conf=0.61
```

### Step 3 of 3: SAM (Segment Anything Model, ViT-H) - box → pixel mask

SAM takes each bounding box as a *prompt* and produces a pixel-level binary mask of the object inside it. Using `multimask_output=True` gives three candidate masks; we pick the one with the highest predicted IoU score.

The result is a `supervision.Detections` object containing:
- `xyxy`: (N, 4) float array of box coordinates
- `confidence`: (N,) float array of detection scores
- `class_id`: (N,) int array mapping each detection to a class in the RAM tag list
- `mask`: (N, H, W) bool array of pixel masks

**Why three models instead of one?**

A single end-to-end detector like YOLO needs both a fixed class list *and* pixel-level annotation for every class at training time. RAM+GroundingDINO+SAM separates these concerns: RAM needs only image-level tags (cheap to collect), GroundingDINO needs only box annotations (abundant), and SAM was trained on 1 billion masks across all object types. Together they cover the full vocabulary of natural language.


In [ ]:
# ── Load models (run once; takes ~30 s on first call) ─────────────────────────
_ram_tf = TS.Compose([
    TS.Resize((384, 384)),
    TS.ToTensor(),
    TS.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Loading RAM...")
ram_model = _ram_cls(pretrained=RAM_CHECKPOINT, image_size=384, vit="swin_l")
ram_model.eval().to(DEVICE)

print("Loading GroundingDINO...")
gd_model = GroundingDINOModel(
    model_config_path=GROUNDING_DINO_CONFIG,
    model_checkpoint_path=GROUNDING_DINO_CHECKPOINT,
    device=DEVICE,
)

print("Loading SAM (vit_h)...")
_sam = sam_model_registry["vit_h"](checkpoint=SAM_CHECKPOINT)
_sam.to(DEVICE)
sam_predictor = SamPredictor(_sam)

print("All detection models loaded ✓")

# ── Detection pipeline ─────────────────────────────────────────────────────────
BOX_THRESH = 0.25
TXT_THRESH = 0.25
NMS_THRESH = 0.50

def detect_frame(rgb: np.ndarray):
    """RAM tagging → GroundingDINO boxes → SAM masks. Returns (sv.Detections, labels, raw_tags)."""
    # 1. RAM
    t_img = _ram_tf(Image.fromarray(rgb)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        tags_str = inference_ram(t_img, ram_model)[0]
    classes = [t.strip() for t in tags_str.replace(" | ", ",").split(",") if t.strip()]

    # 2. GroundingDINO - returns sv.Detections (no phrases tuple)
    detections = gd_model.predict_with_classes(
        image=rgb, classes=classes,
        box_threshold=BOX_THRESH, text_threshold=TXT_THRESH,
    )
    keep = detections.confidence > BOX_THRESH
    detections = detections[keep]
    if len(detections) > 1:
        # sv 0.17: non_max_suppression takes (N,5) array [x1,y1,x2,y2,score]
        preds = np.concatenate([detections.xyxy, detections.confidence[:, None]], axis=1)
        keep_mask = sv.non_max_suppression(preds, iou_threshold=NMS_THRESH)
        detections = detections[keep_mask]
    labels = [
        classes[ci] if ci is not None and ci < len(classes) else "object"
        for ci in (detections.class_id if detections.class_id is not None else [])
    ]

    # 3. SAM
    H, W = rgb.shape[:2]
    masks = []
    if len(detections) > 0:
        sam_predictor.set_image(rgb)
        for box in detections.xyxy:
            m, scores, _ = sam_predictor.predict(box=box, multimask_output=True)
            masks.append(m[np.argmax(scores)])
        detections.mask = np.stack(masks)
    else:
        detections.mask = np.zeros((0, H, W), dtype=bool)

    return detections, labels, tags_str

# ── Run on all frames ──────────────────────────────────────────────────────────
all_detections, all_labels, all_tags = [], [], []

for f in frames:
    t0 = time.time()
    dets, labels, tags = detect_frame(f["rgb"])
    print(f"[{f['episode']} step={f['step']}] {len(dets):2d} objects  "
          f"tags: {tags[:80]}  ({time.time()-t0:.1f}s)")
    all_detections.append(dets)
    all_labels.append(labels)
    all_tags.append(tags)

## Stage 2 - Visualization of detections

Before continuing to 3D processing it is important to visually verify that the detection pipeline produced sensible results. Two helpers from concept-graphs are used:

### `vis_result_fast` - quick annotated image

Overlays bounding boxes and semi-transparent coloured masks on the original RGB image using the `supervision` library's annotators. Each object gets a randomly assigned colour (determined by `instance_random_color=True`) so you can visually distinguish overlapping detections.

**What to look for:**
- Are the boxes tight around the objects, or do they bleed into the background?
- Are there obvious missed objects (things you can see but no mask)?
- Are there spurious detections (masks covering pure background)?

### `show_mask` + `show_box` - detailed single-frame view

For the first frame we draw masks and boxes directly on a matplotlib axes. This shows:
- The **RAM tag string** - the full open-vocabulary caption RAM produced (useful for debugging if GroundingDINO missed something: check whether the tag was in the list)
- The **text prompt** - the subset of tags that actually produced detections
- Individual box labels for each detected object

**Common issues at this stage:**

| Symptom | Likely cause | Fix |
|---|---|---|
| No detections | BOX_THRESH too high, or RAM produced no tags | Lower thresholds or inspect `tags_str` |
| Many duplicate boxes | NMS_THRESH too low | Raise NMS_THRESH toward 0.7 |
| Mask covers wrong region | SAM prompted with imprecise box | Lower BOX_THRESH to get tighter boxes first |


In [ ]:
# ── vis_result_fast: all frames side-by-side ───────────────────────────────────
fig, axes = plt.subplots(1, len(frames), figsize=(5 * len(frames), 5))
if len(frames) == 1:
    axes = [axes]
for ax, f, dets, labels in zip(axes, frames, all_detections, all_labels):
    title = f"{f['episode']}  step={f['step']}"
    if len(dets) == 0:
        ax.imshow(f["rgb"]); ax.set_title(f"{title}  (no detections)", fontsize=8); ax.axis("off")
        continue
    # Give detections sequential class_ids matching our labels list so vis_result_fast
    # can index into it without going out of range
    import dataclasses
    dets_vis = dataclasses.replace(dets)
    dets_vis.class_id = np.arange(len(dets_vis))
    annotated, _ = vis_result_fast(f["rgb"], dets_vis, labels, instance_random_color=True)
    ax.imshow(annotated)
    ax.set_title(f"{title}  {len(dets)} objects", fontsize=8)
    ax.axis("off")
plt.suptitle("Stage 1 result - vis_result_fast", fontweight="bold")
plt.tight_layout(); plt.show()

# ── Detailed view for the first frame (replaces vis_result_slow_caption whose
#    tostring_rgb() was removed in matplotlib ≥ 3.8) ──────────────────────────
f0, d0, l0, t0_tags = frames[0], all_detections[0], all_labels[0], all_tags[0]
if len(d0) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(11, 8))
    ax.imshow(f0["rgb"])
    for i in range(len(d0)):
        show_mask(d0.mask[i], ax, random_color=True)
    for box, label in zip(d0.xyxy, l0):
        show_box(box, ax, label)
    ax.set_title(
        f"Tags: {t0_tags[:100]}\nPrompt: {', '.join(l0[:8])}\n"
        f"{f0['episode']}  step={f0['step']}",
        fontsize=8, fontweight="bold",
    )
    ax.axis("off")
    plt.tight_layout(); plt.show()

## Stage 3 - OpenCLIP visual features

### What is CLIP?

CLIP (Contrastive Language-Image Pre-training) is a model trained to map images and text into a **shared embedding space**. Two embeddings are "close" (high cosine similarity) if the image and text describe the same thing, and "far" otherwise.

**OpenCLIP ViT-H-14** is the largest publicly available CLIP variant:
- Image encoder: Vision Transformer with patch size 14 px, trained on LAION-2B
- Embedding dimension: **1024**
- Trained with supervised contrastive loss on 2 billion image-text pairs

### Why CLIP features for object tracking?

When the robot moves and the same object appears in a new frame from a different angle, pixel-level comparison fails. CLIP embeddings are **view-invariant**: a top-down photo of a red pot and a side-on photo of the same pot will produce similar embeddings because both activate the same semantic concepts ("pot", "red", "cooking vessel").

This is exactly what `MapObjectList` needs: a similarity signal that stays stable across camera viewpoints.

### What `compute_clip_features_batched` does

For each detected object:
1. Crop the RGB image around the bounding box (with 20 px padding)
2. Resize and normalise the crop to the model's input size (224×224)
3. Run a forward pass through the image encoder → 1024-d float32 vector
4. L2-normalise the vector (so cosine similarity = dot product)

Also encodes the text label with the text encoder (for potential text-visual matching, though we primarily use image features).

**Output**: `img_fts` is a `(N, 1024)` numpy array, one row per detection. This is stored in `all_clip_fts` and later attached to each entry in `MapObjectList`.


In [ ]:
print("Loading CLIP (ViT-H-14, laion2b_s32b_b79k)...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-H-14", pretrained="laion2b_s32b_b79k"
)
clip_tokenizer = open_clip.get_tokenizer("ViT-H-14")
clip_model = clip_model.to(DEVICE).eval()
print("CLIP loaded ✓")

all_clip_fts = []   # list of (N_i, 1024) numpy arrays, one per frame

for f, dets, labels in zip(frames, all_detections, all_labels):
    if len(dets) == 0:
        all_clip_fts.append(np.zeros((0, 1024), dtype=np.float32))
        continue
    # class_id must be sequential indices into labels; copy detections to avoid mutation
    import dataclasses
    dets_clip = dataclasses.replace(dets)
    dets_clip.class_id = np.arange(len(dets_clip))
    crops, img_fts, txt_fts = compute_clip_features_batched(
        f["rgb"], dets_clip, clip_model, clip_preprocess, clip_tokenizer, labels, DEVICE
    )
    # returns (image_crops, image_feats_np, text_feats_np) - already numpy arrays
    all_clip_fts.append(img_fts)
    print(f"[{f['episode']} step={f['step']}] CLIP features: {img_fts.shape}")

## Stage 4 - RealSense depth → Open3D 3D point clouds

### Why do we need 3D?

The 2D detection pipeline tells us *what* objects are in the scene and *where* in the image (pixel coordinates). To reason about spatial relationships ("the apple is **inside** the bowl", "the kettle is **on** the stove") we need their 3D positions.

### Camera model: pinhole projection

The RealSense camera follows a standard **pinhole model**. Given a pixel `(u, v)` with measured depth `z` (metres), the 3D position in camera frame is:

```
X = (u - cx) * z / fx
Y = (v - cy) * z / fy
Z = z
```

where `fx=914.27, fy=913.27, cx=647.07, cy=356.33` are the calibrated intrinsics for our 1280×720 sensor. This is called **back-projection**.

For each object, we apply this formula to every pixel inside the SAM mask where the depth is valid (> 0) and stack the resulting `(X, Y, Z)` triplets into an **Open3D PointCloud**.

### Noise removal

Real depth sensors produce noisy readings: flying pixels at object edges, depth holes on reflective surfaces. We run **statistical outlier removal** (`remove_statistical_outlier`): for each point, compute the mean distance to its 10 nearest neighbours; remove it if that mean is more than 2 standard deviations above the global average.

### Oriented bounding box

After denoising we fit an **OrientedBoundingBox** (OBB) to the point cloud. Unlike an axis-aligned box, an OBB can be rotated to tightly wrap an elongated object (e.g. a knife lying diagonally). The OBB is used by `MapObjectList` to compute 3D intersection-over-union (IoU) between detections.

**Example output for a pot on the stove:**
```
PointCloud: 2847 points
OBB center: (0.12, 0.03, 0.84)   ← 84 cm in front of camera
OBB extent: (0.18, 0.15, 0.12)   ← ~18 cm wide, 15 cm deep, 12 cm tall
```


In [ ]:
def frame_to_pcds(f, dets, labels):
    """Backproject detections into 3D using the frame's real depth + intrinsics."""
    depth = f["depth"]          # float32, metres
    rgb   = f["rgb"]
    K     = f["K"]
    H, W  = rgb.shape[:2]

    items = []
    for i, label in enumerate(labels):
        if dets.mask is None or i >= len(dets.mask):
            continue
        mask = dets.mask[i]
        if mask.shape != (H, W) or int(mask.sum()) < 25:
            continue
        pcd = create_object_pcd(depth, mask, K, rgb)
        if len(pcd.points) < 25:
            continue
        cl, idx = pcd.remove_statistical_outlier(nb_neighbors=10, std_ratio=2.0)
        pcd = pcd.select_by_index(idx)
        if len(pcd.points) < 10:
            continue
        bbox = pcd.get_oriented_bounding_box()
        items.append({"label": label, "pcd": pcd, "bbox": bbox, "det_idx": i})
    return items

all_pcds = []
for f, dets, labels in zip(frames, all_detections, all_labels):
    pcds = frame_to_pcds(f, dets, labels)
    all_pcds.append(pcds)
    depth = f["depth"]
    valid = depth[depth > 0]
    print(f"[{f['episode']} step={f['step']}] {len(pcds)} valid point clouds  "
          f"depth range [{valid.min():.2f}, {valid.max():.2f}] m")

# ── Top-down scatter plot (camera-frame XZ plane) ─────────────────────────────
fig, axes = plt.subplots(1, len(frames), figsize=(5 * len(frames), 4))
if len(frames) == 1:
    axes = [axes]
for ax, f, pcds in zip(axes, frames, all_pcds):
    for item in pcds:
        pts = np.asarray(item["pcd"].points)
        ax.scatter(pts[:, 0], pts[:, 2], s=3, color=color_for(item["label"]),
                   alpha=0.6, label=item["label"])
    ax.set_title(f"{f['episode']}  step={f['step']}", fontsize=8)
    ax.set_xlabel("x (m)"); ax.set_ylabel("z (m, forward)"); ax.grid(alpha=0.3)
    ax.set_aspect("equal", adjustable="datalim")
    if pcds:
        handles = [mpatches.Patch(color=color_for(d["label"]), label=d["label"]) for d in pcds]
        ax.legend(handles=handles, fontsize=7, loc="best", ncol=2)
plt.suptitle("Per-frame point clouds - camera-frame top-down view (real depth)", fontweight="bold")
plt.tight_layout()
plt.show()

## Stage 5 - `MapObjectList`: multi-frame 3D object accumulation

This is the **SLAM (Simultaneous Localisation and Mapping) core** of ConceptGraphs. It answers the question: "Have I seen this object before in a previous frame?"

### The data structure

`MapObjectList` is simply a Python list of dictionaries (the "canonical objects"). Each entry holds:

| Key | Type | Meaning |
|---|---|---|
| `pcd` | `o3d.PointCloud` | Accumulated 3D points across all frames |
| `bbox` | `o3d.OrientedBoundingBox` | Tight 3D bounding box recomputed after each merge |
| `clip_ft` | `(1024,)` tensor | Running mean of CLIP embeddings across detections |
| `text_ft` | `(1024,)` tensor | Running mean of text embeddings |
| `label` | `list[str]` | All labels assigned to this object across frames |
| `num_detections` | `int` | How many times this object was detected |
| `inst_color` | `tuple` | RGB colour for visualisation (fixed at first detection) |

### How a new detection is associated

For each new detection `d` and each existing canonical object `o`, two similarity scores are computed:

**Spatial similarity (3D IoU):** the volume overlap between `d.bbox` and `o.bbox` divided by their union. Objects in the same position will have high IoU; an object viewed from the other side of the room will have IoU ≈ 0.

**Visual similarity (CLIP cosine):** the dot product between `d.clip_ft` and `o.clip_ft` (both L2-normalised). A value near 1.0 means the crops look like the same semantic object; near 0 means they look unrelated.

These are combined:
```
agg_sim = (1 + phys_bias) * spatial_sim + (1 - phys_bias) * visual_sim
```
With `phys_bias=0`, spatial and visual similarity are weighted equally.

**Decision rule:**
- If `max(agg_sim[d, :]) > sim_threshold` → merge `d` into the best-matching existing object
- Otherwise → add `d` as a new canonical object

### What "merge" means

When a detection is merged into an existing object:
- The new point cloud is added (`pcd += new_pcd`)
- The combined cloud is voxel-downsampled (to avoid growing indefinitely) and DBSCAN-denoised
- The OBB is recomputed on the merged cloud
- CLIP features are averaged (`clip_ft = mean(old_ft, new_ft)`)
- `num_detections` is incremented

**Example across two frames:**

```
Frame 0:  detect "pot"  → new object [0] (1 detection, 1200 pts)
Frame 1:  detect "pot"  → agg_sim[new, obj0] = 0.73 > 0.5 → merge into [0]
          detect "stove"→ agg_sim[new, obj0] = 0.02 → new object [1]
Result: MapObjectList has 2 canonical objects
  [0] "pot"   - 2 detections, 2100 pts
  [1] "stove" - 1 detection,   980 pts
```

### Note on camera poses

In a full deployment, each frame's point cloud is first transformed into **world frame** using the robot's pose estimate (from GradSLAM, ORB-SLAM, or AprilTag localisation). This means objects seen from different robot positions still accumulate into a single world-frame map.

In this demo we skip pose transformation because the dataset does not provide per-frame localisation. All point clouds stay in camera frame, which means the 3D IoU is only meaningful for frames where the camera has barely moved (which is approximately true for frames sampled from the same short trajectory).


In [ ]:
map_cfg = SimpleNamespace(
    spatial_sim_type="iou_accurate",
    match_method="sim_sum",
    phys_bias=0.0,
    sim_threshold=0.5,
    dbscan_remove_noise=True,
    dbscan_eps=0.10,
    dbscan_min_points=10,
    merge_overlap_thresh=0.8,
    merge_visual_sim_thresh=0.5,
    merge_text_sim_thresh=0.5,
    obj_min_points=0,
    obj_min_detections=0,
    downsample_voxel_size=0.025,
    # fields read by utils.py but not triggered in our demo path
    mask_area_threshold=0.0,
    mask_conf_threshold=0.0,
    max_bbox_area_ratio=1.0,
    min_points_threshold=0,
    skip_bg=False,
    device=DEVICE,
)

objects = MapObjectList(device=DEVICE)

for f_idx, (f, pcds_f, clip_fts, dets, labels) in enumerate(
    zip(frames, all_pcds, all_clip_fts, all_detections, all_labels)
):
    det_list = DetectionList()
    for item in pcds_f:
        di = item["det_idx"]
        if di >= len(clip_fts):
            continue
        ft = F.normalize(torch.from_numpy(clip_fts[di]).float().to(DEVICE), dim=-1)
        det_list.append({
            "pcd":            item["pcd"],
            "bbox":           item["bbox"],
            "clip_ft":        ft,
            "text_ft":        ft,
            "class_id":       [labels.index(item["label"]) if item["label"] in labels else 0],
            "num_detections": 1,
            "inst_color":     color_for(item["label"]),
            "label":          [item["label"]],  # list so merge_obj2_into_obj1 can += it
        })

    if len(det_list) == 0:
        continue

    if len(objects) == 0:
        for d in det_list:
            objects.append(d)
    else:
        spatial_sim = compute_spatial_similarities(map_cfg, det_list, objects)
        visual_sim  = compute_visual_similarities(map_cfg, det_list, objects)
        # spatial_sim is on CPU (from Open3D bboxes); move both to DEVICE
        spatial_sim = spatial_sim.to(DEVICE)
        visual_sim  = visual_sim.to(DEVICE)
        agg_sim     = aggregate_similarities(map_cfg, spatial_sim, visual_sim)
        agg_sim[agg_sim < map_cfg.sim_threshold] = float("-inf")
        objects = merge_detections_to_objects(map_cfg, det_list, objects, agg_sim)

    print(f"Frame {f_idx} [{f['episode']} step={f['step']}]: "
          f"+{len(det_list)} → {len(objects)} canonical objects")

print(f"\n{'='*60}")
print(f"Final map: {len(objects)} canonical objects")
for i, obj in enumerate(objects):
    print(f"  [{i:2d}] {obj.get('label', ['?'])[0]:20s}  "
          f"pts={len(obj['pcd'].points):5d}  n_dets={obj['num_detections']}")

## Stage 6 - LLaVA object captions

### Why not just use the RAM tags?

RAM tags like `"pot"` or `"stove"` are useful for detection but lose fine-grained detail. A more descriptive caption like `"small silver saucepan with black handle"` lets the downstream LLM distinguish between two different pots, understand material properties, or answer questions about the scene.

### LLaVA (Large Language and Vision Assistant)

LLaVA is a multimodal model that combines a vision encoder (CLIP) with an LLM (LLaMA or similar) via a learned projection layer. It takes an image and a text question, and produces a free-form text answer.

We run it locally via **Ollama** - a tool that makes large models available as a local API server (similar to running `docker run` but for LLMs). No API key or internet connection needed.

The prompt we send:
```
"This is a crop centered on one object, with extra context around it.
 Describe only the central object in 3 to 7 words.
 Output only the description, no full sentences or extra punctuation."
```

Constraining to 3-7 words prevents the model from producing verbose descriptions that would confuse the scene graph stage.

### One caption per canonical object per frame

For each canonical object in `MapObjectList`, we find every frame where that object was detected, extract the image crop, and call LLaVA. This gives us multiple independent captions of the same object from different viewpoints. Disagreements between captions (e.g. `"silver pot"` vs `"cooking pot"`) are resolved in Stage 7 by the GPT scene-graph builder.

### The `_crop_with_mask` helper

Before sending to LLaVA, the image crop is prepared by the same helper used in `captioner.py`: the bounding-box region is extracted with 12 px padding, and pixels *outside* the SAM mask are darkened (multiplied by 0.15). This focuses the VLM's attention on the masked object and prevents it from describing the background.


In [ ]:
LLAVA_BACKEND = "llava"   # pulled via: ollama pull llava

obj_captions = {}   # canonical_idx → list[str]
obj_crops    = {}   # canonical_idx → list[PIL.Image]

for obj_idx, obj in enumerate(objects):
    label = obj.get("label", [f"obj_{obj_idx}"])[0]
    captions, crops = [], []

    for f, dets, labels_f in zip(frames, all_detections, all_labels):
        H, W = f["rgb"].shape[:2]
        for di, lbl in enumerate(labels_f):
            if lbl != label:
                continue
            if dets.mask is None or di >= len(dets.mask):
                continue
            mask = dets.mask[di]
            bbox = dets.xyxy[di]
            crop_img = _crop_with_mask(f["rgb"], mask, bbox, mask_mode="none")
            crops.append(crop_img)
            try:
                res = _caption(f["rgb"], mask, bbox, backend=LLAVA_BACKEND, mask_mode="none")
                captions.append(res.text)
            except Exception as e:
                print(f"  [WARN] LLaVA failed for {label}: {e}")
            break   # one observation per frame

    obj_captions[obj_idx] = captions
    obj_crops[obj_idx]    = crops
    cap_str = " | ".join(captions) if captions else "(no captions)"
    print(f"[{obj_idx:2d}] {label:20s}  → {cap_str[:100]}")

# ── Display crops + captions for first N objects ───────────────────────────────
N_SHOW = min(len(objects), 6)
for obj_idx in range(N_SHOW):
    obj   = objects[obj_idx]
    label = obj.get("label", [f"obj_{obj_idx}"])[0]
    crops = obj_crops.get(obj_idx, [])
    caps  = obj_captions.get(obj_idx, [])
    if not crops:
        continue
    fig, axes = plt.subplots(1, len(crops), figsize=(4 * len(crops), 3))
    if len(crops) == 1:
        axes = [axes]
    for ax, crop, cap in zip(axes, crops, caps):
        ax.imshow(crop); ax.axis("off")
        ax.set_title(cap[:40], fontsize=8, wrap=True)
    plt.suptitle(f"[{obj_idx}] {label}  ({len(caps)} LLaVA captions)", fontweight="bold")
    plt.tight_layout(); plt.show()

## Stage 7 - Scene graph: node refinement + spatial edges

### What is a scene graph?

A scene graph is a structured representation of a scene as a **graph** where:
- **Nodes** = objects (each with a canonical label and 3D position)
- **Edges** = spatial or semantic relationships between objects (`"on"`, `"in"`, `"near"`)

For example:
```
apple ──[in]──► bowl
bowl  ──[on]──► countertop
kettle──[near]─ stove
```

This compact representation is directly usable by a robot's task planner: "put the apple in the bowl" translates to checking whether the apple node has an `in` edge to the bowl node.

### Step 7a: node refinement

Each canonical object has a list of LLaVA captions (from Stage 6) and a 3D bounding-box. We send these to GPT (via Portkey) with a structured prompt from `GPTPrompt.system_prompt` - the same prompt used in the ConceptGraphs paper's `build_scenegraph_cfslam.py` script.

GPT is asked to return a JSON object like:
```json
{"object_tag": "silver saucepan with black handle"}
```

The refinement step:
1. Resolves inconsistencies between different captions of the same object
2. Picks a more precise label than the raw RAM tag (e.g. `"pot"` → `"stainless steel pot"`)
3. Filters hallucinations (LLaVA sometimes describes the background)

If GPT returns an unparseable response (or the API call fails), we fall back to the best RAM label.

### Step 7b: spatial edge labelling

We provide GPT with all node tags and their 3D bounding-box centres, and ask it to identify spatial relationships. The LLM uses both semantic knowledge (apples are usually *in* bowls, kettles sit *on* stoves) and the 3D positions (if object A's centre is 30 cm directly above object B, it is likely *on* B).

The output is a JSON array:
```json
[
  {"a": 2, "b": 5, "relation": "on"},
  {"a": 0, "b": 3, "relation": "near"}
]
```

**Why use an LLM instead of pure geometry?**

Pure geometry can tell you that object A is above object B, but not *whether that relationship is meaningful*. A book floating 2 cm above a shelf is `on` the shelf; a picture hanging 50 cm above a table is not `on` the table. The LLM combines geometric evidence with world knowledge to make the right call.


In [ ]:
from conceptgraph.scenegraph.GPTPrompt import GPTPrompt

gpt_prompt = GPTPrompt()

# ── Step 7a: refine node captions with GPT ────────────────────────────────────
REFINE_SYS = gpt_prompt.system_prompt

node_tags = {}   # obj_idx → refined tag string

for obj_idx, obj in enumerate(objects):
    caps = obj_captions.get(obj_idx, [])
    bbox = obj["bbox"]
    center = np.asarray(bbox.center).tolist()
    extent = np.asarray(bbox.extent).tolist()

    user_msg = json.dumps({
        "id":          obj_idx,
        "captions":    caps if caps else [obj.get("label", ["unknown object"])[0]],
        "bbox_center": [round(v, 3) for v in center],
        "bbox_extent": [round(v, 3) for v in extent],
    })

    try:
        text, _ = llm.query(
            prompt={"system": REFINE_SYS, "user": user_msg},
            sampling_params={},
        )
        data = json.loads(text) if isinstance(text, str) else text
        tag  = data.get("object_tag", obj.get("label", ["unknown"])[0])
    except Exception as e:
        print(f"  [WARN] node refine failed for [{obj_idx}]: {e}")
        tag = obj.get("label", [f"obj_{obj_idx}"])[0]

    node_tags[obj_idx] = tag
    print(f"[{obj_idx:2d}] {obj.get('label',['?'])[0]:20s} → {tag}")

# ── Step 7b: spatial edge labelling ──────────────────────────────────────────
EDGE_SYS = (
    "You are a spatial scene-graph builder for indoor robot scenes. "
    "Given a list of objects with their 3D bounding-box centers (x, y, z in metres), "
    "identify meaningful spatial relations. "
    "For EVERY pair (a, b) where a relation exists, output one JSON object with keys "
    "'a', 'b', 'relation' where relation ∈ {\"on\", \"in\", \"near\", \"none\"}. "
    "Only include pairs where relation is not 'none'. "
    "Reply with a JSON array and nothing else."
)

nodes_payload = [
    {"id": i, "tag": node_tags[i],
     "bbox_center": [round(v, 3) for v in np.asarray(objects[i]["bbox"].center).tolist()]}
    for i in range(len(objects))
]

edges = []
try:
    raw, _ = llm.query(
        prompt={"system": EDGE_SYS, "user": json.dumps(nodes_payload)},
        sampling_params={},
    )
    raw = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
    edges = json.loads(raw)
except Exception as e:
    print(f"[WARN] edge labelling failed: {e}")

# ── Rich text display of the scene graph ─────────────────────────────────────
SEP = "=" * 64

print(f"\n{SEP}")
print("  SCENE GRAPH - NODES")
print(SEP)
for i, tag in node_tags.items():
    n_det = objects[i]["num_detections"]
    center = np.asarray(objects[i]["bbox"].center)
    print(f"  [{i:2d}]  {tag:<30s}  "
          f"detections={n_det}  pos=({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}) m")

print(f"\n{SEP}")
print(f"  SCENE GRAPH - EDGES  ({len(edges)} relations)")
print(SEP)
if edges:
    for e in edges:
        a_tag = node_tags.get(e.get("a", -1), "?")
        b_tag = node_tags.get(e.get("b", -1), "?")
        rel   = e.get("relation", "?").upper()
        print(f"  {a_tag}  ──[{rel}]──▶  {b_tag}")
else:
    print("  (no spatial edges found)")

# ── Adjacency list view ────────────────────────────────────────────────────────
if edges:
    print(f"\n{SEP}")
    print("  ADJACENCY LIST  (object → relations)")
    print(SEP)
    from collections import defaultdict
    adj = defaultdict(list)
    for e in edges:
        a_tag = node_tags.get(e.get("a", -1), "?")
        b_tag = node_tags.get(e.get("b", -1), "?")
        rel   = e.get("relation", "?")
        adj[a_tag].append((rel, b_tag))
    for src_tag, rels in sorted(adj.items()):
        print(f"  {src_tag}")
        for rel, tgt in rels:
            print(f"      └─[{rel}]─▶ {tgt}")

print(f"\n{SEP}")

# ── Matplotlib figure: nodes as circles, edges as arrows ──────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

n = len(node_tags)
if n > 0:
    # arrange nodes in a circle
    angles = [2 * np.pi * i / n for i in range(n)]
    R = 3.0
    pos = {i: (R * np.cos(a), R * np.sin(a)) for i, a in enumerate(angles)}

    fig, ax = plt.subplots(figsize=(max(10, n * 1.2), max(8, n * 1.0)))
    ax.set_aspect("equal")
    ax.axis("off")

    RELATION_COLORS = {"on": "#e74c3c", "in": "#3498db", "near": "#2ecc71"}

    # draw edges first (so nodes render on top)
    for e in edges:
        a, b = e.get("a", -1), e.get("b", -1)
        rel  = e.get("relation", "?")
        if a not in pos or b not in pos:
            continue
        x0, y0 = pos[a]
        x1, y1 = pos[b]
        color = RELATION_COLORS.get(rel, "#95a5a6")
        ax.annotate(
            "", xy=(x1, y1), xytext=(x0, y0),
            arrowprops=dict(arrowstyle="-|>", color=color, lw=2.0,
                            connectionstyle="arc3,rad=0.15"),
        )
        mx, my = (x0 + x1) / 2, (y0 + y1) / 2
        ax.text(mx, my, rel, fontsize=8, color=color, ha="center", va="center",
                bbox=dict(fc="white", ec=color, boxstyle="round,pad=0.2", lw=1))

    # draw nodes
    for i, tag in node_tags.items():
        x, y = pos[i]
        c = color_for(tag)
        circle = plt.Circle((x, y), 0.45, color=c, zorder=3)
        ax.add_patch(circle)
        # wrap long labels
        label = tag if len(tag) <= 18 else tag[:16] + "..."
        ax.text(x, y, label, ha="center", va="center", fontsize=7,
                fontweight="bold", color="white", zorder=4,
                wrap=True)
        ax.text(x, y - 0.58, f"[{i}]", ha="center", va="top", fontsize=6,
                color="#555555", zorder=4)

    # legend for edge colours
    legend_patches = [
        mpatches.Patch(color=c, label=rel)
        for rel, c in RELATION_COLORS.items()
    ]
    ax.legend(handles=legend_patches, loc="lower right", fontsize=9,
              title="Relation", title_fontsize=9)

    ax.set_title("Scene Graph", fontsize=14, fontweight="bold", pad=12)
    # auto-scale
    ax.set_xlim(-R - 1.2, R + 1.2)
    ax.set_ylim(-R - 1.2, R + 1.2)
    plt.tight_layout()
    plt.show()

## Stage 8 - Save the map and visualize

### What gets saved

The accumulated `MapObjectList` is serialised to a gzip-compressed pickle file (`scene_map.pkl.gz`). This is the same format used by the concept-graphs scripts, so you can load it in the interactive viewer without re-running the full pipeline.

The file contains:
- `objects`: the list of canonical objects (point clouds, bboxes, CLIP features, labels, captions)
- `node_tags`: the GPT-refined object labels from Stage 7
- `edges`: the spatial edge list from Stage 7

### Inline 3D scatter (matplotlib)

The notebook shows a static matplotlib 3D scatter of all accumulated point clouds, coloured by object. This gives a bird's-eye view of the reconstructed scene. It runs in-kernel so no display is needed.

**Limitations of the inline view:** matplotlib's 3D rendering is approximate (no depth-sorting, no lighting). For serious inspection, use the Open3D viewer below.

### Interactive terminal viewers

The concept-graphs repo ships three terminal scripts that require an **Open3D GUI window** (needs a display or VNC). Run them from a terminal after activating the `conceptgraph` conda env:

```bash
conda activate conceptgraph
source reflect_clean/third_party/concept-graphs/env_vars.bash

# ① Interactive 3D viewer
#    Opens an Open3D window. Click on any object to see its label.
#    Press 'c' to toggle point cloud / bounding-box view.
python $CG_FOLDER/conceptgraph/scripts/visualize_cfslam_results.py \
    --result_path <path/to/scene_map.pkl.gz>

# ② LLaVA-interactive viewer
#    Same as ①, but clicking an object sends its crop to LLaVA
#    and prints the answer in the terminal. Good for live Q&A.
python $CG_FOLDER/conceptgraph/scripts/visualize_cfslam_interact_llava.py \
    --result_path <path/to/scene_map.pkl.gz>

# ③ Animate the mapping process (requires per-frame saved objects)
#    Produces a video of objects appearing frame-by-frame.
python $CG_FOLDER/conceptgraph/scripts/animate_mapping_save.py \
    --input_folder <path/to/objects_all_frames/>
```

The path printed at the end of the save cell can be pasted directly into `--result_path`.


In [ ]:
# ── Save map to disk ───────────────────────────────────────────────────────────
OUT_DIR = analysis_experiment_dir("perception_conceptgraphs_robo2vlm")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_PATH = OUT_DIR / "scene_map.pkl.gz"

objects_to_save = prepare_objects_save_vis(objects, downsample_size=0.025)
with gzip.open(MAP_PATH, "wb") as f:
    pickle.dump({"objects": objects_to_save, "bg_objects": None,
                 "class_colors": {}, "node_tags": node_tags, "edges": edges}, f)
print(f"Map saved → {MAP_PATH}")
print(f"Run: python $CG_FOLDER/conceptgraph/scripts/visualize_cfslam_results.py --result_path {MAP_PATH}")

# ── Static inline 3D scatter (matplotlib, no GPU framebuffer needed) ───────────
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401

fig = plt.figure(figsize=(12, 7))
ax  = fig.add_subplot(111, projection="3d")

for i, obj in enumerate(objects):
    pts  = np.asarray(obj["pcd"].points)
    tag  = node_tags.get(i, obj.get("label", [f"obj_{i}"])[0])
    c    = color_for(tag)
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=1, color=c, alpha=0.5, label=tag)
    center = np.asarray(obj["bbox"].center)
    ax.text(center[0], center[1], center[2], tag[:15], fontsize=7, color="black")

ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Accumulated MapObjectList - 3D scatter (camera frame)", fontweight="bold")
# de-duplicate legend
handles, seen = [], set()
for i, obj in enumerate(objects):
    tag = node_tags.get(i, obj.get("label", [f"obj_{i}"])[0])
    if tag not in seen:
        seen.add(tag)
        handles.append(mpatches.Patch(color=color_for(tag), label=tag))
ax.legend(handles=handles, fontsize=7, loc="upper left", ncol=2)
plt.tight_layout(); plt.show()